In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import tensorflow as tf
import os
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Input, Reshape
import tensorflow.keras.backend as K
import random
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder
from TraceDataGeneration import TraceDataGeneration, TraceDataGenerationCheck

In [ ]:
function = []
for i in range(3):
    function.append(random.choice(function_list))
function = np.array(function)

# data preparation

In [8]:
function_list = ['constant',
                 'impulse_like',
                 'linear_transition',
                 'step_like',
                ]

In [9]:
one = OneHotEncoder()
one.fit(np.array(function_list).reshape(-1,1))

OneHotEncoder()

In [4]:
generator = TraceDataGeneration(n=1, global_range=0, seed=1)
function = []
for _ in range(3):
    function.append(random.choice(function_list))
function = np.array(function)
traces, params = generator.automated_generation_random_para(function)

In [5]:
param_label = np.zeros(shape=(len(params), 6))
for num, param in enumerate(params):
    for key, value in param.items():
        if key == 'start_value':
            param_label[num, 0] = value
        elif key == 'peak_value':
            param_label[num, 1] = value
        elif key == 'end_value':
            param_label[num, 2] = value
        elif key == 't1':
            param_label[num, 3] = value
        elif key == 't2':
            param_label[num, 4] = value
        else:
            param_label[num, 5] = value

In [6]:
function_label = one.transform(function.reshape(-1,1)).toarray()

NameError: name 'one' is not defined

In [ ]:
x = traces[0]

In [96]:
y = np.concatenate([function_label, param_label], axis=1)
y.shape

(3, 10)

In [10]:
X = []
Y = []
for i in range(500):
    generator = TraceDataGeneration(n=1, global_range=0, seed=i)
    function = []
    for _ in range(3):
        function.append(random.choice(function_list))
    function = np.array(function)
    traces, params = generator.automated_generation_random_para(function)
    
    
    param_label = np.zeros(shape=(len(params), 6))
    for num, param in enumerate(params):
        for key, value in param.items():
            if key == 'start_value':
                param_label[num, 0] = value
            elif key == 'peak_value':
                param_label[num, 1] = value
            elif key == 'end_value':
                param_label[num, 2] = value
            elif key == 't1':
                param_label[num, 3] = value
            elif key == 't2':
                param_label[num, 4] = value
            else:
                param_label[num, 5] = value
                
    function_label = one.transform(function.reshape(-1,1)).toarray()
    x = traces[0].reshape(1,-1)
    y = np.concatenate([function_label, param_label], axis=1)
    X.append(x)
    Y.append(y)

C:\Users\HYKP\PycharmProjects\Personal_Research\Deep_generator\TraceDataGeneration.py:803: RuntimeWarning: divide by zero encountered in double_scalars
  m = (end_value - start_value) / (transition_length - 1)
C:\Users\HYKP\PycharmProjects\Personal_Research\Deep_generator\TraceDataGeneration.py:805: RuntimeWarning: invalid value encountered in multiply
  transition = start_value + m * t + noise
C:\Users\HYKP\PycharmProjects\Personal_Research\Deep_generator\TraceDataGeneration.py:803: RuntimeWarning: divide by zero encountered in double_scalars
  m = (end_value - start_value) / (transition_length - 1)
C:\Users\HYKP\PycharmProjects\Personal_Research\Deep_generator\TraceDataGeneration.py:805: RuntimeWarning: invalid value encountered in multiply
  transition = start_value + m * t + noise


In [53]:
check_generator = TraceDataGenerationCheck()
step1 = getattr(check_generator, function[0])(**params[0])
step2 = getattr(check_generator, function[1])(**params[1])
step3 = getattr(check_generator, function[2])(**params[2])

label = check_generator.combine_traces(step1, step2, step3)

In [103]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
  # 텐서플로가 첫 번째 GPU에 1GB 메모리만 할당하도록 제한
  try:
    tf.config.experimental.set_virtual_device_configuration(
        gpus[0],
        [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=2048)])
  except RuntimeError as e:
    # 프로그램 시작시에 가상 장치가 설정되어야만 합니다
    print(e)

In [204]:
for k, y in enumerate(Y):
    Y[k] = y.reshape(1,3,10)
    

In [196]:
def trial_model():
    inputs = Input(shape=120)
    #x = Flatten()(inputs)
    x = Dense(10, activation='relu')(inputs)
    x = Dense(20, activation='tanh')(x)
    outputs = Dense(30, activation='linear')(x)
    outputs = Reshape((3,10))(outputs)
    
    return Model(inputs=inputs, outputs=outputs)
    

In [197]:
model = trial_model()
model.summary()

Model: "model_17"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_23 (InputLayer)        [(None, 120)]             0         
_________________________________________________________________
dense_63 (Dense)             (None, 10)                1210      
_________________________________________________________________
dense_64 (Dense)             (None, 20)                220       
_________________________________________________________________
dense_65 (Dense)             (None, 30)                630       
_________________________________________________________________
reshape_19 (Reshape)         (None, 3, 10)             0         
Total params: 2,060
Trainable params: 2,060
Non-trainable params: 0
_________________________________________________________________


In [198]:
X[0].shape

(1, 120)

In [205]:
model = trial_model()

# model.compile(optimizer='adam',
#               loss='sparse_categorical_crossentropy',
#               metrics=['accuracy'])

optimizer = optimizer = tf.keras.optimizers.Adam()

num_epochs = 5
for epoch in range(num_epochs):
    print(f"Start of epoch {epoch + 1}")
    for x, y in zip(X, Y):
        with tf.GradientTape() as tape:
            pred_y = model(x)
            loss = tf.keras.losses.categorical_crossentropy(y, pred_y)

        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))

Start of epoch 1
Start of epoch 2
Start of epoch 3
Start of epoch 4
Start of epoch 5


In [208]:
model.predict(X[1])

array([[[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]]],
      dtype=float32)

In [209]:
Y[1]

array([[[  0.        ,   1.        ,   0.        ,   0.        ,
          54.34263887,  67.09824983,  62.09824983,  21.        ,
           0.        ,  40.        ],
        [  1.        ,   0.        ,   0.        ,   0.        ,
         100.        ,   0.        , 100.        ,   0.        ,
           0.        ,  40.        ],
        [  0.        ,   0.        ,   1.        ,   0.        ,
          -1.05451016,   0.        ,  10.        ,   3.        ,
          10.        ,  40.        ]]])